# 08-电笔与电子琴项目实战


## 1. 本篇学习范围

本篇对应课程最后一组视频 **59—64**：非接触式电笔原理与绘制、NE555 电子琴原理图、PCB 元件摆放、布局与丝印调整。

两个项目的学习目标不同：

| 项目 | 输入 | 核心电路 | 输出 | 本篇重点 |
| --- | --- | --- | --- | --- |
| 非接触式电笔 | 交流电场通过微小电容耦合形成的信号 | 三极管级联放大 | LED 指示 | 读清级间连接，区分感应区与指示电流区 |
| NE555 电子琴 | 按键选择电阻链的接点 | RC 无稳态振荡 | 方波经耦合电容驱动发声器件 | 算有效电阻与频率，把原理图落实到 PCB |

前置知识：**05-三极管与 MOS 管**、**06-NE555 定时器与振荡电路**、**07-PCB 设计与 EDA 入门**。本篇直接使用这些知识，不再重讲全部基础概念。

读完后应能做到：

1. 逐个网络说明两个项目的连接和作用。
2. 从电子琴的按键接点算出有效定时电阻及理论频率。
3. 说明关键元件为什么这样摆放、哪些走线需要优先处理。
4. 完成原理图、封装、PCB、丝印及低压调试的检查表。

**资料边界：**已核对视频目录和配套 `模拟电路.pdf` 中的两张原理图；视频未逐段观看或转录。PCB 操作原则结合嘉立创 EDA 标准版官方文档整理，计算、调试与器件选型提醒属于学习补充。本篇没有实际制作 PCB，也没有取得实测结果。


## 2. 做项目时，先把“原理—连接—实物”对应起来

画图之前先回答三个问题：电路用什么信号工作，信号经过哪些器件，器件从哪里获得能量。

```text
电笔：变化电场 → 微小电容耦合 → J1 → Q1 → Q2 → Q3 → LED 指示
                                电池为放大电路和 LED 提供能量

电子琴：按键 → 选择 RB → C2 周期充放电 → NE555 输出方波 → C4 → 发声器件
                         电源为振荡器和输出负载提供能量
```

上图是**功能框图**，实际连线以本篇的网络表和原讲义为准。

项目不能只检查“图形看起来一样”。至少要把以下内容对应起来：

| 层次 | 必须对应的内容 | 常见错误 |
| --- | --- | --- |
| 原理 | 输入、供电、控制、输出之间的关系 | 把感应到的能量误当作 LED 的主要供能 |
| 原理图 | 引脚号、网络、元件值、极性 | 线交叉但没有连接；2、6 脚漏连 |
| 封装 | 器件引脚与 PCB 焊盘编号 | 三极管外形相似，B/C/E 却不同 |
| PCB | 铜线、回流、安装及操作位置 | 按键放不下；去耦离芯片太远 |
| 实物 | 元件型号、方向、焊接及负载 | NE555 型号与供电不匹配；喇叭负载过重 |

建议把“照图复现”和“为实物补全”记录为两个版本。补加 LED 限流、改变供电或加入音频驱动时，要同步修改原理图、PCB、BOM 和说明。


## 3. 非接触式电笔：感应片怎样获得信号

讲义第 68—69 页使用三极管放大案例说明非接触式电笔。J1 是金属感应片，作为电路的输入端。

交流导线附近存在随时间变化的电场。导线与 J1 即使没有直接接触，也可能形成很小的耦合电容。变化的电压通过这个电容产生微弱的耦合电流，再由后面的三极管电路增强指示能力。

对于理想电容，有：

$$i=C\frac{dv}{dt}$$

这个关系帮助理解：耦合电容越小，输入通常越弱；电压变化方式、距离和几何位置也会影响结果。**实际有效电容和回路由导线、外壳、人体、周围物体及电路共同决定，不能只用一个固定电容值推算灵敏度。**

需要分清三件事：

- **没有直接电接触**不等于不存在电场耦合。
- 输入信号主要起控制作用，LED 的主要能量来自电池。
- 电池负端是本电路的参考点；便携电笔并没有因此自动连接到大地。

这种简化电路适合学习感应与放大。它对电场、位置、屏蔽和环境敏感，**LED 不亮不能证明线路已经安全断电**，也不能用它确认零线、接地状态或隔离是否可靠。


## 4. 电笔原理图：按网络读清 Q1、Q2、Q3

以讲义第 69 页电路为准，逐条核对：

| 网络或连接 | 讲义中的连接关系 | 作用 |
| --- | --- | --- |
| 输入 | J1 接 Q1 基极 | 把感应信号送入第一级 |
| 第一级输出 | Q1 发射极接 Q2 基极 | 前级驱动第二级 |
| 第二级输出 | Q2 发射极接 Q3 基极 | 第二级驱动末级 |
| 正电源 | Q1、Q2 集电极接电池正端 | 为前两级提供电流 |
| 指示支路 | 电池正端经过 LED1 接 Q3 集电极 | Q3 控制 LED 支路电流 |
| 负电源 | Q3 发射极接电池负端 | 指示支路的返回路径 |

讲义使用 NPN 三极管符号。可以把它理解为**前级发射极驱动后级基极的级联放大**。

**特别注意：三个集电极没有全部共接。**Q1、Q2 的集电极接正电源，Q3 的集电极位于 LED 下方。不要为了套用一个电路名称，把原图的连接改成另一种结构。

读图时沿两条路径检查：

```text
信号驱动：J1 → Q1 的 B → Q1 的 E / Q2 的 B → Q2 的 E / Q3 的 B
指示电流：电池正端 → LED1 → Q3 的 C → Q3 的 E → 电池负端
```

多级电路能提高对微弱输入的响应，但不能把最终 LED 电流简单认定为“输入电流 × 三个固定 β”。β 会随器件、电流和温度变化，漏电、级间偏置、饱和状态及 LED 负载也影响结果。

讲义的放大描述用于帮助理解，实际指示电流应由电路条件和测量确认。


## 5. 电笔原理图绘制与封装核对

绘制时先保持 J1、Q1—Q3、LED1 和电源的关系，再根据实际器件补充参数。

### 原理图顺序

1. 放置三只 NPN 三极管、LED、供电接口与感应片接口。
2. 标清 B、C、E 功能，按上一节网络表连接。
3. 检查 Q1、Q2 集电极的正电源线，以及 Q3 集电极的 LED 支路。
4. 为正、负电源和输入网设置清楚的名称，检查连接点。
5. 记录器件的完整型号，并给实际设计补上需要的限流与电源处理。

### 选封装时要核对什么

| 器件 | 核对项 | 原因 |
| --- | --- | --- |
| Q1—Q3 | 型号、B/C/E 引脚号、封装视角 | 相似封装可能采用不同引脚顺序 |
| LED1 | A/K 极性、额定电流、封装尺寸 | 反接可能不亮；电流需受控 |
| J1 | 金属片形状、安装位置、连接方式 | 会影响耦合与外壳配合 |
| 电池或供电接口 | 电压、极性、插头与焊盘方向 | 图中的电源符号不等于通用实物规格 |
| 附加限流电阻 | 阻值、功率、封装及网络位置 | 必须限制 LED 支路，而不是随意改动整个电源网 |

**课程图没有在这里给出可通用于所有实现的三极管型号、引脚顺序、板尺寸和供电条件。**这些参数需要查实际器件手册，不能依据外形或经验代填。

在封装管理器中核对“符号引脚号 → 焊盘编号”。仅选择一个名字相近的封装，不能保证对应关系正确。


## 6. 电笔 PCB：先处理感应位置，再安排指示支路

电笔的输入非常弱，元件摆放同时影响机械结构和感应效果。

建议的功能分区：

```text
接近感应位置                         方便观察和安装
[ J1 / 金属感应片 ] — [ Q1、Q2 ] — [ Q3、LED 支路 ] — [ 电池接口 ]
```

这只是布局思路，不代表原视频的板形、尺寸或精确坐标。

- **J1 先满足外壳与感应位置。**确认感应片不会裸露到可接触带电部件的位置，再确定连接方式。
- **J1 到 Q1 基极的连接短而清楚。**减少不必要的长支线，避免紧贴 LED 电流路径和干扰源。
- **Q1、Q2 的级间连接紧凑。**保留焊接和检查空间，不为缩小板子把引脚挤到无法确认。
- **Q3、LED 与电源回路便于检查。**限制指示电流，并避免较大电流路径穿过输入区。
- **保持输入区域清洁。**焊剂残留、潮湿和表面漏电可能改变工作状态。

感应区不能机械照搬“整板铺 GND”的经验。**在 J1 附近或其下方增加接地铜，可能改变耦合或产生屏蔽，降低灵敏度。**是否留空、铺铜及感应片尺寸需要结合结构验证，不能只看 DRC。

修改感应区域后，记录几何结构与指示变化。环境变化很大时，不能把一次成功响应当成稳定的检测性能。


## 7. 电笔的 LED 电流与低压验证

讲义图没有明确画出独立 LED 限流电阻。实际制作前应检查指示支路的限流方式，以及 LED、三极管和电池的额定条件。

一种学习补充的处理是：

```text
电源正端 → R_LED → LED → Q3 集电极
Q3 发射极 → 电源负端
```

只是在**LED 支路**加入限流，不能误把 Q1、Q2 的集电极也改接到 LED 后面。

若末级充分导通并用饱和压降近似，可估算：

$$R_{LED}\approx\frac{V_{CC}-V_F-V_{CE(sat)}}{I_{LED}}$$

**计算练习，非课程实测参数：**假设供电 5 V、LED 正向压降 2.0 V、Q3 饱和压降 0.2 V，选 680 Ω，则：

$$I_{LED}\approx\frac{5-2.0-0.2}{680}\approx4.1\text{ mA}$$

$$P_R\approx I_{LED}^2R\approx11.5\text{ mW}$$

实际选择要检查最大供电、LED 压降范围、阻值公差和功率余量。**弱输入时 Q3 可能没有饱和，上式不能保证实际亮度。**加入电阻后也应重新验证响应。

### 验证顺序

1. 断电检查极性、B/C/E、焊接以及 LED 限流支路。
2. 用符合器件要求的低压电源，设置合理限流，先确认待机电流和器件温升。
3. 如需检查级联功能，可使用隔离的低压交流信号源，通过受控的小电容耦合输入；避免直接驱动过大基极电流。
4. 记录输入条件、LED 电流及各级节点的变化。测量探头本身可能影响高阻输入，应考虑这种影响。
5. 比较是否接入感应片、感应片位置及环境变化时的响应。

以上低压练习验证的是级联响应，**不能据此证明市电检测能力或断电判断可靠性**。本篇不安排裸露市电试验。

| 现象 | 优先检查 |
| --- | --- |
| 始终不亮 | 电源、LED 极性、Q3 接法、感应连接、器件脚号 |
| 始终亮 | 输入区污染或漏电、焊桥、器件损坏、环境耦合 |
| 靠近后偶尔亮 | 距离、方向、屏蔽、外壳与参考耦合条件 |
| 加铺铜后不灵敏 | 是否屏蔽了输入区、改变了耦合结构 |
| LED 或三极管发热 | 限流与极性、支路短路、额定值；先断电处理 |


## 8. NE555 电子琴：把完整图拆成五个模块

讲义第 81 页说明声音与频率，第 82 页给出 NE555 电子琴电路。本项目用八个按键选择不同定时电阻，改变方波频率，再由发声器件产生不同音高。

| 模块 | 讲义元件 | 功能 |
| --- | --- | --- |
| 供电与去耦 | J3、C3 | 给 U1 和输出支路供电，减小电源波动 |
| 定时核心 | U1、R1、C2 | 完成电容充放电和输出翻转 |
| 音高选择 | S1—S8、串联电阻链 | 选择 7 脚到定时节点之间的有效 RB |
| 控制脚处理 | RESET 接 VCC、C1 接 CONT | 保持正常工作，处理控制端 |
| 发声输出 | 3 脚 OUT、C4、LS1 | 通过串联电容把交流变化送到发声器件 |

按脚号检查连接：

| NE555 引脚 | 图中接法 |
| --- | --- |
| 1 GND | 接电源负端 |
| 2 TRIG | 与 6 脚相连，接定时电容 C2 上端 |
| 3 OUT | 经 C4 接 LS1，LS1 另一端接地 |
| 4 RESET | 接正电源 |
| 5 CONT | 经 C1 接地 |
| 6 THRES | 与 2 脚共用定时节点 |
| 7 DISCH | 接 R1 下端与各按键共用端 |
| 8 VCC | 接正电源，与 C3 去耦对应 |

`104` 表示 **100 nF = 0.1 μF**。本图 C1、C2、C3 都标为 `104`，但用途不同：C1 为控制端旁路，C2 为定时电容，C3 为电源去耦。

讲义其他标准振荡示例中，控制端电容使用过 10 nF；不要为了让两张示例图数值相同而擅自改图。实际设计可以按对应器件手册选择，并记录修改。

J3 在原图标注 **3—9 V**。这不能直接证明所有 NE555 都能在 3 V 工作；普通双极型 NE555 应核对供电下限。学习复现可先按**具体型号手册支持的 5 V 供电**设计，器件、负载和丝印也要与这个版本一致。


## 9. 八个按键怎样选择电阻：先找接点，再求和

图中 R1 = 1 kΩ，位于正电源与 7 脚之间，记为 $R_A$。按键决定的是 7 脚到 C2 定时节点之间的 $R_B$。

电阻链从 S1 的远端向 C2 方向依次为：

```text
S1 接点 — R3 2k — S2 接点 — R4 2k — S3 接点 — R6 2k — S4 接点
        — R7 2k — S5 接点 — R8 2k — S6 接点 — R9 2k — S7 接点
        — R2 1k — S8 接点 — R5 10k — C2 定时节点（2、6 脚）

S1—S8 的另一组触点均接 7 脚。
```

按下一个按键，相当于把 7 脚接到电阻链的相应抽头。**只有这个接点到定时节点之间的电阻参与 RB 求和**；接点外侧的悬空链段不计入单键的 RB。

以下按单键闭合、理想接触、$R_A=1$ kΩ、$C_2=100$ nF 计算：

| 按键 | 有效 RB 的组成 | RB | 理论频率 | OUT 高电平占空比 |
| --- | --- | --- | --- | --- |
| S1 | R3+R4+R6+R7+R8+R9+R2+R5 | 23 kΩ | 307.0 Hz | 51.06% |
| S2 | R4+R6+R7+R8+R9+R2+R5 | 21 kΩ | 335.5 Hz | 51.16% |
| S3 | R6+R7+R8+R9+R2+R5 | 19 kΩ | 369.9 Hz | 51.28% |
| S4 | R7+R8+R9+R2+R5 | 17 kΩ | 412.2 Hz | 51.43% |
| S5 | R8+R9+R2+R5 | 15 kΩ | 465.4 Hz | 51.61% |
| S6 | R9+R2+R5 | 13 kΩ | 534.3 Hz | 51.85% |
| S7 | R2+R5 | 11 kΩ | 627.3 Hz | 52.17% |
| S8 | R5 | 10 kΩ | 687.0 Hz | 52.38% |

在这张图中，S1 的有效电阻最大、音调最低；S8 的有效电阻最小、音调最高。元件编号不是音名，也不必按 R2、R3 的数字大小重新排序。

### 松键和同时按键

- 全部松开时，7 脚与定时电阻链之间没有按键通路，标准无稳态的完整充放电路径被断开；不应继续用上表预测持续振荡。松键瞬间可能出现过渡声，具体行为与初始状态和实际器件有关。
- 多键同时闭合，会把多个抽头一起接到 7 脚，旁路部分链段。理想情况下由**更靠近定时节点的闭合抽头**决定剩余 RB。例如 S1 与 S8 同按，剩余主要是 R5，而不是 23 kΩ 与 10 kΩ 简单并联。
- 这里只有一个振荡器。多键同按不会自动输出多个独立音高，不属于多声部电子琴。


## 10. 电子琴频率计算：把 06 的公式用到项目中

在单键闭合、符合标准无稳态连接且忽略非理想因素时：

$$t_H=\ln(2)(R_A+R_B)C$$

$$t_L=\ln(2)R_BC$$

$$T=\ln(2)(R_A+2R_B)C,\qquad f=\frac{1}{T}$$

$$D_H=\frac{R_A+R_B}{R_A+2R_B}$$

电容充电经过 RA 与 RB；放电主要经过 RB 和 7 脚的内部放电开关。这里的 C 是 **C2**，不是 C1、C3 或输出耦合 C4。

### S1：最长电阻链

$R_A=1$ kΩ，$R_B=23$ kΩ，$C=100$ nF：

$$t_H\approx1.664\text{ ms},\quad t_L\approx1.594\text{ ms}$$

$$T\approx3.258\text{ ms},\quad f\approx307.0\text{ Hz}$$

### S8：只剩 R5

$R_B=10$ kΩ：

$$t_H\approx0.762\text{ ms},\quad t_L\approx0.693\text{ ms}$$

$$T\approx1.456\text{ ms},\quad f\approx687.0\text{ Hz}$$

由此可以预测：按键从 S1 变到 S8，电阻变小、周期缩短、音高上升。

### 参数变化怎样影响声音

- C2 增大，所有按键的理论频率整体下降；C2 减小，整体上升。
- 某段链电阻增大，包含这段的按键频率下降；其余按键未必受同样影响。
- R1 不只是一个装饰性电阻。它限制 7 脚放电开关闭合时从电源来的电流，也影响高电平时间，不能为调音随意短接。
- 器件阈值、放电管压降、电阻电容公差、电源波动和输出负载都会造成实测偏差。

上表给出的是**公式计算值**。调试时应记录实测频率，再与计算值比较，不能把计算值写成已经测到的结果。


## 11. 八个不同音高不等于准确的标准音阶

课程电路能展示“电阻改变 → 周期改变 → 音高改变”。但这组等步长电阻不保证产生准确的 C 大调或其他标准音阶。

原因是：

$$f\propto\frac{1}{R_A+2R_B}$$

频率与有效电阻不是线性关系，而十二平均律的相邻半音频率比为：

$$\frac{f_{n+1}}{f_n}=2^{1/12}\approx1.05946$$

例如课程参数中 $f_{S2}/f_{S1}\approx1.093$，不能据此把 S1、S2 直接认作一对标准全音。

### 学习补充：从目标频率反求有效电阻

保持 RA 与 C 不变：

$$R_B=\frac{1}{2}\left[\frac{1}{\ln(2)fC}-R_A\right]$$

以 440 Hz 为目标，取 $R_A=1$ kΩ、$C=100$ nF：

$$R_B\approx15.894\text{ k}\Omega$$

这是**该按键需要的总 RB**，不是随意把电阻链上的某只电阻改为 15.894 kΩ。

若要做连续八键 C4—C5 标准音阶，可先算每个目标的总 RB，再求相邻总 RB 的差值，形成新电阻链：

| 目标音 | 目标频率 | 所需总 RB，约 |
| --- | --- | --- |
| C4 | 261.63 Hz | 27.071 kΩ |
| D4 | 293.66 Hz | 24.064 kΩ |
| E4 | 329.63 Hz | 21.384 kΩ |
| F4 | 349.23 Hz | 20.155 kΩ |
| G4 | 392.00 Hz | 17.902 kΩ |
| A4 | 440.00 Hz | 15.894 kΩ |
| B4 | 493.88 Hz | 14.106 kΩ |
| C5 | 523.25 Hz | 13.286 kΩ |

例如最低音与下一音之间的链段约需 $27.071-24.064=3.007$ kΩ；最高音到定时节点的基础电阻约需 13.286 kΩ。其余段同理求差。

这些是新的设计目标，不是课程原图数值。实物还需选择可购阻值、考虑公差并校准；共享链段会同时影响多个按键，调某一段后要复查相关音高。若改动参数，应保留原课程复现版，并给调音版单独标记版本。


## 12. 发声支路：C4 的作用与喇叭负载

第 82 页图中：U1 的 3 脚 OUT 经 **C4 = 4.7 μF** 接 LS1，LS1 标注 **8 Ω、0.5 W**，另一端接地。

C4 是串联耦合电容，用于隔离稳态直流分量，让输出的交流变化作用于发声器件。它不是定时电容，不能代入 NE555 的振荡公式。

必须分清：

- **隔离直流**不等于没有负载电流，变化的方波仍会产生电流。
- **8 Ω**是喇叭的标称阻抗，实际阻抗随频率变化。
- **0.5 W**是发声器件的功率规格，不表示电路一定输出 0.5 W。
- 耦合电容不会增加 NE555 的输出驱动能力；增加音量需要同时检查芯片、供电、负载与驱动方案。

### 为什么可能有声，但声音不理想

作为学习近似，若把喇叭暂看作 8 Ω 电阻，并忽略输出阻抗，则 C4 与负载形成的高通转折频率约为：

$$f_c\approx\frac{1}{2\pi RC}=\frac{1}{2\pi\times8\times4.7\times10^{-6}}\approx4.2\text{ kHz}$$

这远高于本图各键的基频，因此低频分量可能明显衰减；方波的高频成分仍可能发声。**真实喇叭并非纯电阻，此式只是说明电容与负载会共同影响音量和音色，不能据此预测精确响度。**

实际复现应查输出电流和功耗条件，避免把数据手册某个最大电流数字当作持续、无失真驱动能力。需要更稳定的发声效果时，可设计适合负载的音频驱动级，再重新核对耦合和供电。

若 C4 使用有极性电容，应依据两端的直流偏置及器件手册确认方向。此接法 OUT 端通常具有正的平均直流电压，喇叭端直流参考为地；仍需检查启动过程、额定电压及所选器件条件。

替换发声器件时也要区分：**有源蜂鸣器内部自带振荡，不能直接当作任意音高的转换器**；无源蜂鸣器或其他发声器件需核对驱动方式与电压。换器件后要修改 BOM、封装和调试预期。


## 13. 电子琴 BOM 与封装：把课程值和实物规格分开

下表记录讲义第 82 页的标注，方便对照原图。封装、完整型号、耐压及功率要由实际设计补全。

| 位号 | 课程标注 | 数量 | 实物核对重点 |
| --- | --- | --- | --- |
| U1 | NE555 | 1 | 具体型号、供电范围、输出条件、封装与 1 脚方向 |
| R1、R2 | 1 kΩ | 2 | 阻值、公差、封装；两者作用不同 |
| R3、R4、R6、R7、R8、R9 | 2 kΩ | 6 | 链段顺序、阻值及焊接位置 |
| R5 | 10 kΩ | 1 | 所有单键都包含的基础 RB |
| C1、C2、C3 | 104，即 100 nF | 3 | 功能、耐压；C2 的实际值影响音高 |
| C4 | 4.7 μF | 1 | 极性、耐压、尺寸及耦合效果 |
| S1—S8 | 按键 | 8 | 触点内部连接、脚距、操作方向和间距 |
| LS1 | 8 Ω、0.5 W | 1 | 阻抗、尺寸、安装或外接方式、驱动要求 |
| J3 | 3—9 V 供电标注 | 1 | 接口极性与实际供电版本；不能直接照抄为器件允许范围 |

### 按键最容易出现的封装错误

常见四脚轻触开关通常包含两个内部互通的端组，按下后把两组接通。**应按所选型号手册或断电通断测量确认分组，不能仅凭外形猜“相邻脚”或“同侧脚”一定怎样连接。**

把两根网络接到同一个内部互通组，可能导致“没有按也始终导通”；脚距不匹配则可能无法安装。

### 元件选择的实用原则

- C2 是定时元件，应选择公差和稳定性满足目标的电容，必要时测量实际值。
- C3 应靠近供电脚，不能因为 BOM 中同为 100 nF 就把其位置当成任意。
- 输出若增加驱动级，BOM 必须增加对应元件，并复查供电电流和接口。
- 音高调校版要更新所有链段值，不能只在文字中写“已经调音”。

保留课程位号有利于学习对照；正式工程如需重新编号，要确保原理图、PCB、BOM 和装配图共同更新。


## 14. 电子琴 PCB 元件摆放：先让手能按，再让电路好连

视频 62—64 的主题包括元件摆放、布局、丝印与 PCB 布局。本节将这些主题整理成可执行的布局顺序，具体界面和板形以所用软件与设计为准。

### 第一步：固定人机与机械位置

1. 先确定板框、安装孔、八个按键的排列和操作间距。
2. 预留指尖按压空间；考虑按键高度、外壳开孔及板边距离。
3. 放置供电接口、发声器件或喇叭接口，避免插头、导线挡住按键。
4. 实物需要孔位或固定件时，先按实际尺寸设置。

### 第二步：围绕 U1 放置关键小回路

- C3 靠近 U1 的 8 脚与 1 脚，缩短电源去耦回路。
- C2 靠近 2、6 脚，减少定时节点的长线和无关分支。
- C1 靠近 5 脚及参考地。
- R1 靠近 VCC 与 7 脚；R5 的定时节点端尽量靠近 C2。
- 电阻链沿按键抽头的逻辑关系安排，避免连线来回绕行。
- C4 与输出接口靠近输出路径，给负载回流留出清楚的路线。

布局示意仅表达功能分区：

```text
[ S1 ][ S2 ][ S3 ][ S4 ][ S5 ][ S6 ][ S7 ][ S8 ]   ← 操作区
[       与抽头对应的电阻链                 ]
                                [ R5、C2、U1 ]   ← 定时区
                                [ C1、C3、R1 ]
[ 供电接口 ]                    [ C4、输出接口 ] ← 电源与发声区
```

并非所有器件都要排得一样整齐。先满足操作、安装、去耦和定时路径，再调整美观与走线长度。摆放完成后，检查每个元件是否可以焊接、返修和识别方向。


## 15. 电子琴布线：区分电源、定时、按键与输出回流

先查看飞线，确认所有网络完整；再按照电路功能安排走线。

| 网络 | 布线重点 | 错误后的可能表现 |
| --- | --- | --- |
| VCC、GND | 宽度与电流需求匹配；去耦路径短，参考连接完整 | 电压波动、复位异常、发热 |
| C2 与 2/6 脚定时节点 | 短、清楚；避开输出大电流及长并行线 | 音高不稳、误触发 |
| 7 脚、R1 与按键共用端 | 接点明确，确认按键共用线没有断开 | 全键无声或部分键异常 |
| 电阻链抽头 | 按原图拓扑连接，不按外观随意交换 | 音高次序错乱、多个键同音 |
| 3 脚、C4 与负载 | 满足实际电流条件，控制输出回流路径 | 电源被拉低、失真、干扰定时 |
| 4、5 脚 | RESET 接正电源，CONT 旁路连到可靠参考地 | 不振荡或状态异常 |

**重点是完整回路。**喇叭电流会经电源及参考地返回；要避免它经过与定时区共用的狭窄、高阻抗连接而引起参考电压变化。

有连续地平面时，仍应考虑回流的自然路径和器件位置；没有地平面时，检查地线是否被绕得很长。不要仅为“分区”任意割裂地参考，导致回流绕路。

### 铺铜后再核对

1. 确认铜区网络正确。
2. 移动元件或修改走线后，重新生成铺铜。
3. 检查焊盘是否真正连接、是否出现孤岛及不期望的细颈。
4. 重新执行连接检查和 DRC。

第 07 篇的铺铜流程可以沿用，但电笔感应区与电子琴普通地参考的需求不同。电笔的 J1 区域需考虑耦合与屏蔽，不能直接复制电子琴的整板铺铜做法。


## 16. 丝印与布局调整：让装配和操作少出错

完成电气布局后再整理丝印；必要时因丝印空间调整器件，但调整后要复查走线、铺铜和规则。

| 对象 | 建议标记 | 核对要点 |
| --- | --- | --- |
| U1 | 位号、1 脚或缺口方向 | 与实际封装方向一致，安装后仍可辨认 |
| LED | 位号、极性 | 对应 A/K，避免只靠图形猜方向 |
| C4 | 位号及需要的极性 | 与所选有极性电容一致 |
| J3 | VCC、GND 或 +/−，实际供电范围 | 与连接器方向及设计版本一致 |
| S1—S8 | 按键编号、低音到高音方向 | 课程参数版可以标音高顺序，不应直接标成精确标准音名 |
| 输出接口 | 信号、返回端、负载用途 | 防止与供电接口混淆 |
| 板信息 | 项目名、版本号、必要日期 | 便于原理图、PCB、BOM 对照 |

丝印不能压在焊盘开口或被关键器件完全遮住。保证文字方向和大小可读，给连接器附近留出极性标记。

如果使用经过调校的标准音阶版，音名应与对应按键、实测调校记录和版本匹配。仅修改丝印音名不会改变电阻链，也不会让音高自动正确。

电笔还应能辨认感应端、LED 方向和供电极性，避免装配时把 J1 当成普通电源接口。


## 17. 原理图转 PCB 后的项目检查

沿用第 07 篇流程，但检查内容针对两个项目具体展开。

### 原理图与封装

- 电笔：J1→Q1、Q1 E→Q2 B、Q2 E→Q3 B；Q1/Q2 C 接正电源；Q3 C 接 LED 支路。
- 电子琴：2 与 6 共接 C2；R1 位于 VCC—7；按键共用端接 7；R5 通向定时节点。
- 4 脚接正电源，5 脚旁路、1/8 脚供电及 C3 正确。
- B/C/E、IC 脚号、LED、电容、开关触点和连接器与焊盘编号对应。
- 参数、器件额定值、供电版本和输出负载已经核对。

### PCB 连通与规则

- 检查未连接网络、短路、错接和焊盘连接。
- 按实际厂家能力设置线宽、间距、孔径及板边规则，执行 DRC。
- 改动原理图后导入 PCB 变更，复查可能保留的旧走线。
- 移动元件后检查关键回路，重建铺铜，再检查一次。
- 检查按键操作空间、接口极性、器件方向和丝印。

**连接正确、DRC 通过、电路功能正确是三个不同层面的结果。**DRC 不会替你发现 NE555 供电不足、音阶不准、LED 缺少有效限流或感应区被屏蔽。

### 制造资料

确认最终版本后，再导出 Gerber 与钻孔并查看预览：板框、铜层、阻焊、丝印与孔位应符合设计。如果还需装配，BOM、坐标、装配说明也要与同一 PCB 版本对应。

此处是提交前的学习检查流程，本笔记未生成真实制造包，也未下单。


## 18. 电子琴低压调试：先看电源，再看定时波形

采用具体器件支持的低压电源，例如经过核对的 5 V 版本。不要把图中“3—9 V”标签当作无需检查的通用条件。

### 调试顺序

1. **断电检查：**极性、短路、封装方向、按键触点分组和链电阻顺序。
2. **先减小输出负担：**初次检查可先不接喇叭或使用经确认合适的测试负载，避免负载问题掩盖振荡问题。
3. **核对供电：**检查 8—1 脚电压、4 脚复位电平和静态电流，出现明显发热先断电。
4. **单键测试振荡：**按 S1，观察 2/6 节点与 3 脚 OUT；再按 S8 对比。
5. **核对阈值：**定时电容应在接近 $1/3 V_{CC}$ 与 $2/3 V_{CC}$ 之间往复变化。5 V 下理想参考约为 1.67 V 与 3.33 V，实际值存在偏差。
6. **逐键记录：**记录频率、周期、占空比与稳定性，确认单键顺序从低到高。
7. **接入实际发声支路：**观察供电、频率、温升和声音是否发生变化，确认负载没有明显扰乱振荡。
8. **检查松键与多键：**记录过渡声和抽头旁路行为；不要把多键当成多声部功能验收。

示波器或频率计的探头会带来负载。测定时节点时采用合适探头；仪器地连接应与这个隔离低压测试电路的参考方式一致。

### 实测记录模板

本表尚未进行测量，“待记录”不是缺省实测值。

| 按键 | 课程参数理论频率 | 实测频率 | 相对偏差 | 供电与声音情况 |
| --- | --- | --- | --- | --- |
| S1 | 307.0 Hz | 待记录 | 待计算 | 待记录 |
| S2 | 335.5 Hz | 待记录 | 待计算 | 待记录 |
| S3 | 369.9 Hz | 待记录 | 待计算 | 待记录 |
| S4 | 412.2 Hz | 待记录 | 待计算 | 待记录 |
| S5 | 465.4 Hz | 待记录 | 待计算 | 待记录 |
| S6 | 534.3 Hz | 待记录 | 待计算 | 待记录 |
| S7 | 627.3 Hz | 待记录 | 待计算 | 待记录 |
| S8 | 687.0 Hz | 待记录 | 待计算 | 待记录 |

$$\text{相对偏差}=\frac{f_{测量}-f_{理论}}{f_{理论}}\times100\%$$

| 现象 | 优先检查 |
| --- | --- |
| 全部按键无振荡 | 电源、RESET、2/6 连接、C2、R1 与 7 脚、按键共用线 |
| 部分键无声或与邻键相同 | 开关触点分组、焊桥、电阻链断线与抽头错接 |
| 所有频率一起偏高或偏低 | C2 实际值与单位、RA、共用 R5、器件条件 |
| 只有包含某链段的键异常 | 对应链电阻及抽头网络 |
| OUT 有波形但喇叭无声 | C4、负载接口、发声器件类型、驱动和音量条件 |
| 接喇叭后音高波动或芯片发热 | 电源能力、去耦、回流、输出负载和功耗 |
| 不按键也持续振荡 | 开关端组误接、焊桥、按键共用线意外接到抽头 |
| 多键按下只剩一个音高 | 对照抽头旁路关系；单振荡器不会自动产生和弦 |


## 19. 项目练习与交付清单

练习按“说明得清楚 → 图纸一致 → 能验证”推进。以下是建议完成项，不代表已经替用户做过实物实验。

### 练习 A：电笔原理图与布局复核

1. 独立画出第 69 页网络，不依赖元件图形的位置记忆。
2. 在图上圈出输入区、级间驱动和 LED 电流支路。
3. 根据实际器件补全型号、封装和指示限流，并说明与讲义的差别。
4. 提供布局说明：J1 位置、输入连线、感应区铺铜取舍、供电与 LED 路径。
5. 做低压功能验证记录，注明环境和测量方法；不能写成安全断电认证。

### 练习 B：课程电子琴复现

1. 保留原图位号与数值，完成原理图和封装核对。
2. 算出 S1—S8 的 RB 和频率，检查顺序。
3. 确定具体供电、负载实现、板框、按键间距与接口。
4. 完成 PCB、丝印、连接检查和 DRC，记录规则设置。
5. 对每键记录实测值；若实际增加驱动，完整记录修改。

### 选做：标准音阶版本

选择明确目标音高，反算总 RB，求相邻链段，再选阻值与调校。保存为独立版本，复查八个键，而不是只调一个键。

| 应保留的资料 | 内容 |
| --- | --- |
| 项目说明 | 目标、原课程参数、实际供电与修改点 |
| 原理图 | 位号、网络、完整参数及器件型号 |
| 封装核对记录 | 脚号、焊盘、尺寸、极性与开关端组 |
| PCB 与检查记录 | 板框、布局、走线、铺铜、连接与 DRC |
| BOM 与装配信息 | 与 PCB 同版本的型号、数量、方向 |
| 制造资料，若需要 | 最终 Gerber、钻孔及预览核对结果 |
| 调试记录 | 理论与实测对照、环境、负载、异常和修正 |

项目完成不能只以“板子画完”作为判断。电笔要能解释响应及局限；电子琴要能解释音高、波形和负载行为，且图纸与实物版本一致。


## 20. 一页复习表

| 主题 | 必须记住的结论 |
| --- | --- |
| 电笔输入 | J1 通过电场与微小电容耦合得到信号，不直接接到市电 |
| 电笔供能 | LED 的主要能量来自电池 |
| 级间连接 | Q1 E→Q2 B，Q2 E→Q3 B |
| 集电极区别 | Q1/Q2 C 接正电源；Q3 C 经 LED 支路接正电源 |
| 指示限流 | 实物需核对限流；讲义简图不能自动保证额定电流 |
| 感应区布局 | 位置、污染与铺铜会影响耦合，不能机械照搬普通铺铜 |
| 电笔指示局限 | 不亮不能证明安全断电 |
| 电子琴定时节点 | 2、6 脚共接 C2；C2=104=100 nF |
| RA | R1=1 kΩ，位于 VCC—7 脚 |
| RB | 单键抽头到定时节点之间的串联总电阻 |
| 音高顺序 | S1：23 kΩ、约 307 Hz；S8：10 kΩ、约 687 Hz |
| 频率公式 | $f=1/[\ln(2)(R_A+2R_B)C]$ |
| 音阶 | 不同频率不等于准确标准音阶，等电阻步长不等于等音程 |
| 多键 | 抽头会被共同连接、部分链段被旁路；单振荡器不会自动输出和弦 |
| C1、C3、C4 | 分别为控制端旁路、电源去耦、输出耦合；都不是定时 C2 |
| 发声负载 | C4 隔离直流，不增强驱动；8 Ω/0.5 W 是负载规格 |
| 摆放顺序 | 按键和结构先确定，随后安排 U1、定时、去耦与输出回路 |
| 丝印 | 极性、1 脚、接口、键序与实际供电版本一致 |
| DRC | 验证规则，不保证音高、灵敏度、供电或输出能力正确 |
| 版本一致 | 原理图、PCB、BOM、制造资料和调试记录要对应 |

课程知识衔接：01—04 提供电路、器件、测量和计算基础；05 提供晶体管知识；06 解释 NE555；07 介绍 EDA 与 PCB；08 将这些知识用于两个综合项目。课程目录已核对到视频 64，本篇作为这组资料的项目收尾。


## 21. 自测题与参考答案

### 先独立作答

1. 非接触电笔没有与导线直接连接，为什么仍可能产生响应？LED 的主要能量从哪里来？
2. 讲义第 69 页三个三极管的集电极是否全部接在同一网络？
3. 电笔输入区铺了大量接地铜之后灵敏度下降，可能是什么原因？
4. LED 不亮是否能证明线路已经安全断电？
5. 电笔指示支路在 5 V、LED 压降 2.0 V、Q3 饱和压降 0.2 V、680 Ω 下，估算电流是多少？这个值能保证弱输入时的亮度吗？
6. 电子琴 S5 按下时有效 RB 是多少？C2=100 nF、RA=1 kΩ 时频率约为多少？
7. 为什么 R1 不应直接短接？
8. S1 与 S8 同时按下，是否应把 23 kΩ 和 10 kΩ 作为两只完整电阻并联计算？
9. C1、C2、C3 都标 104，是否可以忽略它们的位置和功能？C4 能否代入定时公式？
10. 八个按键都产生不同声音，是否说明已经得到准确的标准音阶？
11. 目标 440 Hz、RA=1 kΩ、C=100 nF 时，需要的总 RB 约为多少？
12. OUT 波形正常但喇叭声音很弱，应检查哪些方面？
13. 按键不按也导通，封装和触点方面首先检查什么？
14. DRC 没有错误，但所有频率偏高约一倍，首先检查什么？
15. 改了原理图电阻和 PCB 元件位置后，提交制造资料前还需要做什么？

### 参考答案

1. 变化电场通过微小电容耦合形成输入信号，三极管级联增强响应；LED 主要由电池供能。
2. 不是。Q1、Q2 集电极接电池正端，Q3 集电极经 LED 支路接正端。
3. 接地铜可能屏蔽感应片或改变耦合结构；也要检查感应片位置、污染、连接和环境。
4. 不能。屏蔽、距离、耦合、器件及电池状态都可能使指示失效。
5. 约 $(5-2.0-0.2)/680=4.1$ mA。这是充分导通并采用近似压降时的估算，弱输入时 Q3 未必饱和，实际亮度要测量。
6. S5 的 RB=R8+R9+R2+R5=15 kΩ，频率约 465.4 Hz。
7. 7 脚放电开关闭合时，R1 限制从正电源来的电流；它也影响充电时间和占空比。
8. 不是。两个抽头被共同接到 7 脚，之间的链段被旁路；理想情况下剩余 RB 主要为 R5=10 kΩ。
9. 不能。C1 为控制端旁路、C2 为定时、C3 为电源去耦；C4 是输出耦合，不能代替公式中的 C2。
10. 不能。频率与 RB 呈非线性关系，这组电阻值没有保证标准音程；需要确定目标、计算并校准。
11. 总 RB 约 15.894 kΩ，需要按总阻值及链段关系分配，不能任意替换一只电阻。
12. C4 数值与连接、喇叭或蜂鸣器类型、阻抗和安装、NE555 输出条件、电源能力及驱动设计；不要只靠减小电阻追求音量。
13. 核对开关两个内部互通端组、符号脚号与焊盘映射，并检查焊桥。
14. 先核对 C2 是否误装成约 50 nF、单位是否理解正确，再检查共用电阻和测量方法；共同参数可能使所有键一起偏移。
15. 导入变更，复查旧走线与网络，重建铺铜，检查结构/丝印、连通和 DRC，再导出并预览同版本制造资料，同时更新 BOM 和说明。


## 22. 与原课程资料的对应关系

| 课程视频 | 本篇对应内容 |
| --- | --- |
| `59_非接触式电笔原理.mp4` | 电场耦合、三极管级联、LED 指示与网络理解 |
| `60_非接触测电笔绘制.mp4` | 电笔原理图、封装、感应区布局与检查 |
| `61_电子琴原理图绘制.mp4` | NE555 引脚、RC 定时、按键电阻链及发声支路 |
| `62_电子琴的pcb元件摆放.mp4` | 按键结构、U1、定时、去耦与输出器件的摆放 |
| `63_布局和丝印调整.mp4` | 极性、接口、音键、版本与丝印空间 |
| `64_电子琴的pcb布局.mp4` | 网络布线、回流、铺铜、规则与最终复核 |

配套 `ppt/模拟电路.pdf` 共 84 页：

- 第 **68—69 页**：非接触式电笔与三极管放大案例；第 69 页是本篇电笔连接表的依据。
- 第 **81—82 页**：声音与频率、NE555 电子琴；第 82 页是按键、电阻链、引脚和元件值的依据。
- 第 83—84 页为演奏示例，复习电路无需复制其中乐谱或歌词。
- PCB 内容依视频目录主题及官方 EDA 文档整理，未声称配套 PDF 给出了完整 PCB 操作过程。

原资料：[百度网盘课程分享](https://pan.baidu.com/s/1AZwoEpfx_2anpJ8p9ofD-Q?pwd=9988)，提取码 `9988`。

### 必要补充参考

- [TI NE555 产品与数据手册入口](https://www.ti.com/product/NE555)：供电、阈值、复位、放电及输出条件；具体实现按对应厂商与完整型号手册核对。
- [嘉立创 EDA 封装管理器](https://docs.lceda.cn/cn/Schematic/Footprint-Manager/)：封装绑定和引脚映射。
- [生成 PCB](https://docs.lceda.cn/cn/Schematic/Convert-to-PCB/)与[导入变更](https://docs.lceda.cn/cn/PCB/Import-Changes/)：转换、同步与旧走线复核。
- [铺铜](https://docs.lceda.cn/cn/PCB/Copper-Pour/)与[设计规则检查](https://docs.lceda.cn/cn/PCB/Design-Rule-Check/)：网络、重建和规则检查。
- [生成制造文件](https://docs.lceda.cn/cn/PCB/Gerber-Generate/)：Gerber、钻孔及制造预览。

本篇依据已核对的课程目录与 PDF 在线预览整理，**视频未逐段观看或转录**。LED 限流示例、频率与音高反算、耦合电容近似、布局建议、器件核对、调试和自测为学习补充。未执行的实验明确用“理论”“预期”或“待记录”表示。

EDA 补充文档按标准版阅读，实际菜单与操作可能随版本不同。本篇交付的是复习笔记；没有实际制作电路、制板、下单或取得测量结果。
